# P08 Цикл обучения и шаг

**Рабочий ноутбук**. Полное объяснение: [локальный урок P08](../../course/pilot/P08.md). Этот файл содержит исполняемый практикум.

Начните с [порядка самостоятельной работы](../../course/self-study.md). Локальный учебник открывается файлом START_HERE.html в корне курса. Если ноутбук скопирован в submissions и относительная ссылка не открывается, найдите урок по пути course/pilot/P08.md от корня репозитория.

## Цель

Выполнить обновление параметров и распознать расходимость.

**Перед началом:** P07: производная и частные производные; P02: MSE.

## Подготовка

Запускайте из корня репозитория в его виртуальном окружении. Первичная установка зависимостей описана в README. После установки выполнение не использует сеть, платные API или GPU.

In [ ]:
from pathlib import Path
import sys
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().resolve()
while not (ROOT / "src" / "mlcourse" / "pilot.py").is_file():
    if ROOT == ROOT.parent:
        raise RuntimeError("Откройте Jupyter из корня ml-llm-course")
    ROOT = ROOT.parent
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))
from mlcourse.pilot import (
    mse, predict_line, linear_gradient, central_difference, fit_standardizer,
    matvec, fit_gradient, make_dataset, run_experiment, predict_payload,
)
plt.rcParams.update({"figure.figsize": (7.2, 4.2), "font.size": 11,
                     "axes.spines.top": False, "axes.spines.right": False})
FIGURES = ROOT / "artifacts" / "figures"
FIGURES.mkdir(parents=True, exist_ok=True)
print("Данные: искусственные учебные примеры. CPU. Состояние ученика не оценивается автоматически.")

## Разобранный пример

Обновление w_next = w − learning_rate × gradient делает небольшой шаг против роста ошибки. Для g(w)=(w−3)² из w=0 шаг 0.1 даёт 0.6, затем 1.08. Большой шаг может перескочить минимум и увеличить ошибку. Для нескольких параметров обновляем каждый его производной.

### Один параметр и три шага обучения

In [ ]:
fig, ax = plt.subplots()
for rate, style, color in [(0.1, "-", "#1f5a99"), (0.5, "--", "#333333"), (1.1, ":", "#b56c19")]:
    w_scalar = 0.; losses = []
    for step in range(18):
        losses.append((w_scalar-3)**2)
        w_scalar -= rate * 2*(w_scalar-3)
    ax.plot(np.arange(len(losses)), np.maximum(losses, 1e-14), linestyle=style, color=color, label=f"Шаг {rate}")
ax.set_yscale("log")
ax.set(xlabel="Номер шага", ylabel="Ошибка, логарифмическая шкала", title="Шаг обучения меняет поведение")
ax.legend(); fig.tight_layout(); fig.savefig(FIGURES / "P08_learning_rate.png", dpi=130); plt.show()
print("Нулевые значения отображены на уровне 1e-14 только для логарифмического графика.")

### Обучить линейную модель

In [ ]:
x, y = [0, 1, 2], [1, 3, 5]
dw, db = linear_gradient(x, y, 0, 0)
print("Первый шаг:", -0.1*dw, -0.1*db)
result = fit_gradient(x, y, learning_rate=0.1, steps=500)
print({k:v for k,v in result.items() if k != "history"}, "final MSE:", result["history"][-1])
assert result["history"][-1] < 1e-10
bad = fit_gradient(x, y, learning_rate=10)
assert bad["status"] == "diverged"
print("Большой шаг:", bad["status"], "записано значений:", len(bad["history"]))

## Практика с поддержкой

Восстановите вручную два первых шага scalar-примера. В линейном примере объясните, почему оба параметра меняются одновременно.

## Самостоятельная задача

Для x=[0,1], y=[2,5], w=b=0 рассчитайте градиент и один шаг с learning_rate=0.1. Затем реализуйте собственный цикл с историей ошибки и остановкой при нечисловых значениях или резком росте. CLI может проверить linear_gradient из вашего файла: --task gradient.

Сначала сохраните собственную попытку. Затем откройте подсказки в локальном уроке и сверяйтесь с эталоном; отмечайте, какая помощь потребовалась. После полного разбора выполните новый вариант без подсказок. При желании можно передать работу на разбор.

In [ ]:
# Здесь ваша реализация или расчёт.
# Ячейка намеренно не содержит готового решения и не ломает запуск примеров.

**Ваше объяснение:** замените этот абзац своим рассуждением, результатом и ограничением. Это поле для ответа ученика, а не пропущенное объяснение автора.

## Проверка

Градиент выведен из функции ошибки и проверен центральной разностью; первый шаг рассчитан из исходных параметров и learning_rate; обновление использует минус; история содержит реальную ошибку; расходимость распознаётся и не объявляется успешным обучением.

Успешный Run All подтверждает запуск подготовленных примеров. Для самопроверки сравните свою работу с критериями и эталоном, объясните шаги своими словами и решите новый вариант. Запишите результат в progress/progress.md относительно корня курса. Самооценка не является независимым подтверждением навыка.

## Следующий шаг

Измените только learning_rate. Объясните, почему больше шаг не обязательно быстрее приводит к решению.

[Основной источник](https://developers.google.com/machine-learning/crash-course/linear-regression/hyperparameters). Все значения в примерах синтетические. Сохраняйте исходные ноутбуки; для своей работы используйте копию.